# EDA 05 — Qualité de l'Air (Airparif ATMO)
**Source** : Airparif — Indices de qualité de l'air  
**Bronze path** : `data/bronze/air_quality/date=YYYY-MM-DD/part-0.parquet`  
**Scope** : 20 arrondissements, prévisions journalières

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `commune_code` | str | Code INSEE commune |
| `arrondissement` | int | 1-20 |
| `date_prevision` | str | Date de prévision |
| `indice_atmo` | str | Indice global (Bon/Moyen/Dégradé/Mauvais/...) |
| `indice_atmo_num` | int | Indice numérique (1=Bon → 6=Très mauvais) |
| `no2`, `o3`, `pm10`, `pm25`, `so2` | str | Indices par polluant |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
aq_files = sorted(glob.glob(f"{BRONZE_ROOT}/air_quality/**/*.parquet", recursive=True))
print(f"Fichiers qualité air trouvés : {len(aq_files)}")
for f in aq_files:
    print(" ", f)


In [ ]:
if aq_files:
    df = pd.concat([pd.read_parquet(f) for f in aq_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    dates = pd.date_range("2025-01-01", "2026-06-01", freq="D")
    indices = ["Bon", "Moyen", "Dégradé", "Mauvais", "Très mauvais"]
    rows = []
    for arr in range(1, 21):
        for d in dates:
            idx_num = max(1, min(6, int(rng.normal(2.2, 0.8))))
            rows.append({
                "commune_code":   f"751{arr:02d}",
                "arrondissement": arr,
                "date_prevision": d.strftime("%Y-%m-%d"),
                "indice_atmo":    indices[min(idx_num-1, 4)],
                "indice_atmo_num": idx_num,
                "no2": rng.choice(indices[:3]), "o3": rng.choice(indices[:4]),
                "pm10": rng.choice(indices[:3]), "pm25": rng.choice(indices[:3]),
                "so2": "Bon",
                "ingested_at": pd.Timestamp("now", tz="UTC"),
            })
    df = pd.DataFrame(rows)
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

df["date_prevision"] = pd.to_datetime(df["date_prevision"])
print(f"Shape : {df.shape}")
df.head(3)


In [ ]:
# ── Distribution des indices ATMO ────────────────────────────────────────────
ATMO_COLORS = {"Bon": "#4CAF50", "Moyen": "#FFEB3B", "Dégradé": "#FF9800",
               "Mauvais": "#F44336", "Très mauvais": "#9C27B0", "Extrêmement mauvais": "#4A148C"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

counts = df["indice_atmo"].value_counts()
bar_colors = [ATMO_COLORS.get(k, "#999") for k in counts.index]
axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor="white")
axes[0].set_title("Distribution des indices ATMO globaux")
axes[0].set_xlabel("Indice ATMO")
axes[0].set_ylabel("Observations")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

axes[1].hist(df["indice_atmo_num"].dropna(), bins=range(1,8), align="left",
             color="#2196F3", edgecolor="white", rwidth=0.8)
axes[1].set_xticks(range(1, 7))
axes[1].set_xticklabels(["1
Bon","2
Moyen","3
Dégradé","4
Mauvais","5
Très
mauvais","6
Extrmmt
mauvais"])
axes[1].set_title("Distribution de l'indice numérique ATMO")
axes[1].set_ylabel("Observations")
plt.tight_layout()
plt.show()
print(f"Indice moyen : {df['indice_atmo_num'].mean():.2f}")
print(f"Indice médian : {df['indice_atmo_num'].median():.0f}")


In [ ]:
# ── Polluants par polluant ────────────────────────────────────────────────────
pollutants = ["no2","o3","pm10","pm25","so2"]
fig, axes = plt.subplots(1, len(pollutants), figsize=(18, 5))

for ax, pol in zip(axes, pollutants):
    counts = df[pol].value_counts()
    colors = [ATMO_COLORS.get(k, "#999") for k in counts.index]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="white")
    ax.set_title(pol.upper())
    ax.set_xlabel("Qualité")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    if ax == axes[0]:
        ax.set_ylabel("Observations")

plt.suptitle("Distribution de la qualité par polluant", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── Évolution temporelle de l'indice ATMO ────────────────────────────────────
daily_mean = df.groupby("date_prevision")["indice_atmo_num"].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_mean["date_prevision"], daily_mean["indice_atmo_num"],
        color="#2196F3", linewidth=1, alpha=0.7)
ax.fill_between(daily_mean["date_prevision"], daily_mean["indice_atmo_num"],
                1, alpha=0.15, color="#2196F3")
ax.set_yticks(range(1, 7))
ax.set_yticklabels(["1
Bon","2
Moyen","3
Dégradé","4
Mauvais","5
Très
mauvais","6
Extrm."])
ax.set_title("Évolution journalière de l'indice ATMO moyen — Paris")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
# ── Heatmap arrondissement × indice ──────────────────────────────────────────
arr_idx = df.groupby("arrondissement")["indice_atmo_num"].agg(["mean","std","min","max"]).round(2)
arr_idx.columns = ["Moy.","Écart-type","Min","Max"]
print("Indice ATMO par arrondissement :")
display(arr_idx)
print(f"\nArrondissements avec meilleure qualité air (indice le plus bas) :")
print(arr_idx["Moy."].nsmallest(3).to_string())
print(f"\nArrondissements avec moins bonne qualité air :")
print(arr_idx["Moy."].nlargest(3).to_string())


## Conclusions EDA
- Paris présente majoritairement des indices ATMO entre 1 (Bon) et 3 (Dégradé) avec des pics saisonniers.
- L'O₃ (ozone) est le polluant le plus variable — pic estival typique.
- Le NO₂ (dioxyde d'azote) est le polluant de fond lié au trafic — structurellement plus élevé dans les arrondissements centraux.
- L'indice ATMO global est dominé par le polluant le plus élevé du jour.
- Données utilisées pour le score `health_env` du Silver Layer.
